# Pipeline de Sanitização, Exclusão de Dias Sem Pregão e Filtro de Liquidez
**Autor:** Pedro Reis  
**Contexto:** Correção de Linhas Vazias Globais e Seleção de Ativos Líquidos

Este script realiza a sanitização avançada da matriz de cotações temporais. O fluxo foi corrigido para tratar anomalias de calendário antes da avaliação de liquidez:
1. **Expulsão de Dias Sem Pregão:** Eliminação de linhas em que *todos* os ativos estão nulos (finais de semana e feriados).
2. **Filtro de Presença em Pregão:** Cálculo exato da liquidez individual baseado apenas nos dias de mercado aberto, retendo ativos com **>= 95%** de presença.
3. **Tratamento de Missing Values Residuais:** Aplicação de *Forward Fill* ($ffill$) e *Backward Fill* ($bfill$).
4. **Persistência Estruturada:** Gravação da matriz final higienizada.

In [3]:
import os
from pathlib import Path
import pandas as pd
import logging

# Configuração do sistema de monitoramento analítico
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] - %(message)s")

# Definição das rotas de arquivos
caminho_entrada = r"C:\VSCodeWorkspace\TCC_Final\data\dados_economatica_tratados\dados_combinados_tcc.csv"
caminho_saida = r"C:\VSCodeWorkspace\TCC_Final\data\Matriz_Precos\Matriz_precos_sanitizada.csv"

# Criação automatizada do diretório de destino
Path(caminho_saida).parent.mkdir(parents=True, exist_ok=True)

print(f"[-] Configuração de rotas estruturada com sucesso.")

[-] Configuração de rotas estruturada com sucesso.


## Ingestão e Purgação de Linhas Nulas Globais (Dias Sem Pregão)

Nesta etapa, removemos as linhas que representam datas em que o mercado financeiro não operou. Utilizamos o método `dropna(how='all')` que elimina a linha de forma estrita apenas se **todas as colunas de ativos** apresentarem valor missing (`NaN`) simultaneamente naquele dia.

In [4]:
print("[+] Carregando a matriz combinada de preços...")

try:
    # Ingestão forçando a indexação cronológica correta
    df_bruto = pd.read_csv(caminho_entrada, index_col='Data', parse_dates=True)
    
    print(f"[Diagnóstico Bruto]:")
    print(f"      -> Dimensões originais: {df_bruto.shape[0]} linhas (datas) x {df_bruto.shape[1]} colunas (ativos)")
    
    # -------------------------------------------------------------------------
    # PASSO NOVO: EXPULSÃO DE DIAS SEM PREGÃO GLOBAL (Feriados / Finais de Semana)
    # -------------------------------------------------------------------------
    # O argumento how='all' garante que só caem as linhas onde 100% dos ativos são NaN
    df_precos_ativos = df_bruto.dropna(how='all')
    
    linhas_removidas = df_bruto.shape[0] - df_precos_ativos.shape[0]
    total_linhas_efetivas = df_precos_ativos.shape[0]
    
    print(f"\n[Ajuste de Calendário]:")
    print(f"      -> Dias sem pregão/feriados globais eliminados: {linhas_removidas} linhas.")
    print(f"      -> Dias efetivos de mercado aberto para análise: {total_linhas_efetivas} linhas.")

    # -------------------------------------------------------------------------
    # APLICAÇÃO DO FILTRO DE LIQUIDEZ SOBRE OS DIAS EFETIVOS
    # -------------------------------------------------------------------------
    limiar_minimo = 0.95
    
    # Agora o cálculo reflete a presença real sobre os dias em que a B3 de fato abriu
    proporcao_presenca = df_precos_ativos.notna().sum() / total_linhas_efetivas
    
    ativos_elegiveis = proporcao_presenca[proporcao_presenca >= limiar_minimo].index.tolist()
    ativos_excluidos = proporcao_presenca[proporcao_presenca < limiar_minimo].index.tolist()
    
    print(f"\n[Filtro de Liquidez Corrigido]:")
    print(f"      -> Ativos aprovados (>= 95% de presença real): {len(ativos_elegiveis)}")
    print(f"      -> Ativos eliminados (< 95% de presença real): {len(ativos_excluidos)}")
    
    # Filtragem dimensional da matriz pelas colunas elegíveis
    df_filtrado_liquidez = df_precos_ativos[ativos_elegiveis].copy()

except Exception as e:
    logging.error(f"[ERRO] Falha crítica no processamento adaptativo: {str(e)}")

[+] Carregando a matriz combinada de preços...
[Diagnóstico Bruto]:
      -> Dimensões originais: 4174 linhas (datas) x 496 colunas (ativos)

[Ajuste de Calendário]:
      -> Dias sem pregão/feriados globais eliminados: 207 linhas.
      -> Dias efetivos de mercado aberto para análise: 3967 linhas.

[Filtro de Liquidez Corrigido]:
      -> Ativos aprovados (>= 95% de presença real): 135
      -> Ativos eliminados (< 95% de presença real): 361


## Imputação Temporal por Propagação e Consistência Final

Após limparmos os dias em que a B3 não funcionou e filtrarmos apenas as ações líquidas, realizamos o *Forward Fill* ($ffill$) e o *Backward Fill* ($bfill$). Esta rotina agora atuará exclusivamente sobre lacunas residuais legítimas (como os dias em que um papel específico ficou temporariamente retido em leilão ou não teve negócios, apesar de a bolsa estar aberta).

In [5]:
print("\n[+] Iniciando imputação de lacunas residuais por propagação temporal...")

# 1. Propagação do último preço negociado para frente
df_sanitizado = df_filtrado_liquidez.ffill()

# 2. Ajuste marginal retrospectivo para o início da série
df_sanitizado = df_sanitizado.bfill()

nulos_finais = df_sanitizado.isna().sum().sum()
print(f"[OK] Matriz de preços nominais higienizada com sucesso.")
print(f"      -> Missing values (NaNs) remanescentes: {nulos_finais}")


[+] Iniciando imputação de lacunas residuais por propagação temporal...
[OK] Matriz de preços nominais higienizada com sucesso.
      -> Missing values (NaNs) remanescentes: 0


## Exportação da Matriz Higienizada

Gravamos em disco o arquivo resultante, que servirá de insumo livre de distorções estruturais de calendário para o cálculo das matrizes estatísticas.

In [6]:
print(f"[+] Salvando arquivo final em: {caminho_saida}")

try:
    df_sanitizado.to_csv(caminho_saida, index=True)
    print("\n--- PIPELINE EXECUTADO COM SUCESSO ---")
    print(f"Matriz Pronta para Retornos: {df_sanitizado.shape[0]} Datas x {df_sanitizado.shape[1]} Ativos.")

except Exception as e:
    logging.critical(f"[ERRO] Falha na gravação final: {str(e)}")

[+] Salvando arquivo final em: C:\VSCodeWorkspace\TCC_Final\data\Matriz_Precos\Matriz_precos_sanitizada.csv

--- PIPELINE EXECUTADO COM SUCESSO ---
Matriz Pronta para Retornos: 3967 Datas x 135 Ativos.


"A fim de mitigar o risco de liquidez e evitar distorções estatísticas decorrentes do preenchimento excessivo de lacunas temporais, estabeleceu-se um critério de corte por Presença em Pregão. Foram elegíveis para compor o universo amostral desta pesquisa apenas os ativos que apresentaram cotações reais registradas em, no mínimo, 95% do total de dias úteis compreendidos no horizonte temporal (2010-2025). O cálculo da métrica de presença foi efetuado de forma estritamente prévia a qualquer rotina de imputação de dados, garantindo que o posterior preenchimento por propagação temporal (Forward Fill) atuasse de forma estritamente residual (máximo de 5% de lacunas por ativo), preservando a fidedignidade da volatilidade e das covariâncias históricas."

================================================================================
APÊNDICE B – SCRIPT DE FILTRAGEM POR LIQUIDEZ E SANITIZAÇÃO DE PREÇOS NOMINAIS
================================================================================

Este apêndice apresenta a documentação técnica e o código-fonte desenvolvido para 
a etapa de sanitização e refinamento do universo amostral de ativos. A rotina aqui 
descrita atua diretamente sobre a matriz de preços combinada, aplicando critérios 
estritos de elegibilidade por liquidez antes da formulação matemática dos portfólios.

Para garantir a robustez estatística das matrizes de covariância e evitar o viés 
causado por ativos de baixíssima negociabilidade (ações cujos preços seriam 
repetidos artificialmente por longos períodos), o algoritmo executa um Filtro 
de Presença em Pregão. A métrica avalia a razão entre os dias de negociação real 
(dados não-nulos de fechamento) e o total de datas mapeadas na janela temporal:

    P_i = \frac{\sum_{t=1}^{T} I(P_{i,t} \neq \text{NaN})}{T}

Onde P_i representa a presença percentual do ativo i, T é o número total de dias 
úteis da série cronológica comum, e I é uma função indicadora que assume valor 1 
se há cotação registrada e 0 caso contrário. Foram mantidos na pesquisa apenas 
os ativos que apresentaram P_i >= 0,95 (95% de presença).

O pipeline de execução foi estruturado sob os seguintes passos:
1. Carga da matriz unificada e diagnóstico da dimensionalidade bruta das séries.
2. Purgação de linhas vazias globais (dias sem pregão na B3) utilizando o critério 
   de exclusão condicional estrito por vetor horizontal, eliminando datas em que 
   todos os ativos apresentavam ausência concomitante de cotação.
3. Cálculo da proporção de nulos individuais por ativo com base exclusiva no 
   denominador de dias úteis efetivos de mercado aberto.
4. Segregação dos ativos e aplicação do filtro de liquidez limitante (P_i >= 0,95).
5. Imputação de dados ausentes residuais por propagação temporal (Forward Fill 
   e Backward Fill) para conferir continuidade perfeita à matriz dimensional.
6. Persistência final em disco no formato de texto delimitado (CSV).

Abaixo encontra-se a implementação integral do notebook correspondente:

[INSERIR O CÓDIGO DAS CÉLULAS 2, 4, 6 e 8 DO PASSO ANTERIOR NESTE ESPAÇO]


import os
from pathlib import Path
import pandas as pd
import logging

# Configuração do sistema de monitoramento analítico
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] - %(message)s")

# Definição das rotas de arquivos
caminho_entrada = r"C:\VSCodeWorkspace\TCC_Final\data\dados_economatica_tratados\dados_combinados_tcc.csv"
caminho_saida = r"C:\VSCodeWorkspace\TCC_Final\data\Matriz_Precos\Matriz_precos_sanitizada.csv"

# Criação automatizada do diretório de destino
Path(caminho_saida).parent.mkdir(parents=True, exist_ok=True)

print(f"[-] Configuração de rotas estruturada com sucesso.")

print("[+] Carregando a matriz combinada de preços...")

try:
    # Ingestão forçando a indexação cronológica correta
    df_bruto = pd.read_csv(caminho_entrada, index_col='Data', parse_dates=True)
    
    print(f"[Diagnóstico Bruto]:")
    print(f"      -> Dimensões originais: {df_bruto.shape[0]} linhas (datas) x {df_bruto.shape[1]} colunas (ativos)")
    
    # -------------------------------------------------------------------------
    # PASSO NOVO: EXPULSÃO DE DIAS SEM PREGÃO GLOBAL (Feriados / Finais de Semana)
    # -------------------------------------------------------------------------
    # O argumento how='all' garante que só caem as linhas onde 100% dos ativos são NaN
    df_precos_ativos = df_bruto.dropna(how='all')
    
    linhas_removidas = df_bruto.shape[0] - df_precos_ativos.shape[0]
    total_linhas_efetivas = df_precos_ativos.shape[0]
    
    print(f"\n[Ajuste de Calendário]:")
    print(f"      -> Dias sem pregão/feriados globais eliminados: {linhas_removidas} linhas.")
    print(f"      -> Dias efetivos de mercado aberto para análise: {total_linhas_efetivas} linhas.")

    # -------------------------------------------------------------------------
    # APLICAÇÃO DO FILTRO DE LIQUIDEZ SOBRE OS DIAS EFETIVOS
    # -------------------------------------------------------------------------
    limiar_minimo = 0.95
    
    # Agora o cálculo reflete a presença real sobre os dias em que a B3 de fato abriu
    proporcao_presenca = df_precos_ativos.notna().sum() / total_linhas_efetivas
    
    ativos_elegiveis = proporcao_presenca[proporcao_presenca >= limiar_minimo].index.tolist()
    ativos_excluidos = proporcao_presenca[proporcao_presenca < limiar_minimo].index.tolist()
    
    print(f"\n[Filtro de Liquidez Corrigido]:")
    print(f"      -> Ativos aprovados (>= 95% de presença real): {len(ativos_elegiveis)}")
    print(f"      -> Ativos eliminados (< 95% de presença real): {len(ativos_excluidos)}")
    
    # Filtragem dimensional da matriz pelas colunas elegíveis
    df_filtrado_liquidez = df_precos_ativos[ativos_elegiveis].copy()

except Exception as e:
    logging.error(f"[ERRO] Falha crítica no processamento adaptativo: {str(e)}")

print("\n[+] Iniciando imputação de lacunas residuais por propagação temporal...")

# 1. Propagação do último preço negociado para frente
df_sanitizado = df_filtrado_liquidez.ffill()

# 2. Ajuste marginal retrospectivo para o início da série
df_sanitizado = df_sanitizado.bfill()

nulos_finais = df_sanitizado.isna().sum().sum()
print(f"[OK] Matriz de preços nominais higienizada com sucesso.")
print(f"      -> Missing values (NaNs) remanescentes: {nulos_finais}")

print(f"[+] Salvando arquivo final em: {caminho_saida}")

try:
    df_sanitizado.to_csv(caminho_saida, index=True)
    print("\n--- PIPELINE EXECUTADO COM SUCESSO ---")
    print(f"Matriz Pronta para Retornos: {df_sanitizado.shape[0]} Datas x {df_sanitizado.shape[1]} Ativos.")

except Exception as e:
    logging.critical(f"[ERRO] Falha na gravação final: {str(e)}")